# EcoPackAI

## Importing Libraries

In [76]:
import pandas as pd
import numpy as np
import openpyxl
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import joblib

## Loading the Dataset

In [77]:
df = pd.read_csv("dataset/EcoPackAI_Final_6.csv")

## Data Preprocessing

In [78]:
df_db = df.rename(columns={
    "Material_Type": "material_type",
    "Product_Type": "product_type",
    "Industry": "industry",
    "Strength (1-10)": "strength",
    "Weight Capacity (kg)": "weight_capacity",
    "Biodegradability Score (1-10)": "biodegradability_score",
    "Recyclability (%)": "recyclability",
    "Cost (USD)": "cost",
    "CO2 Emission (kg CO2/kg)": "co2_emission"
})

df_db.to_csv("dataset/ecoPackAI_db.csv", index=False)

In [79]:
print("Preview of Dataset")
display(df.head())

Preview of Dataset


,Material_Type,Product_Type,Industry,Strength (1-10),Weight Capacity (kg),Biodegradability Score (1-10),Recyclability (%),Cost (USD),CO2 Emission (kg CO2/kg)
0,HDPE Plastic,Mailer Bag,Logistics,9,4.28,3,66.15,2.71,4.55
1,PP Plastic,Fertilizer Bag,Agriculture,9,8.70,1,58.32,2.72,4.31
2,Corrugated Cardboard,Heavy Crate,Logistics,5,95.75,9,75.33,1.04,0.82
3,Bioplastic (PLA),Snack Packaging,Food & Beverage,5,23.46,8,71.52,2.61,0.34
4,Bagasse Fiber,Produce Bag,Agriculture,3,1.30,10,82.88,1.83,0.64


In [80]:
# Info about columns and datatypes
print("\nDataset Info:")
print(df.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10360 entries, 0 to 10359
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Material_Type                  10360 non-null  object 
 1   Product_Type                   10360 non-null  object 
 2   Industry                       10360 non-null  object 
 3   Strength (1-10)                10360 non-null  int64  
 4   Weight Capacity (kg)           10360 non-null  float64
 5   Biodegradability Score (1-10)  10360 non-null  int64  
 6   Recyclability (%)              10360 non-null  float64
 7   Cost (USD)                     10360 non-null  float64
 8   CO2 Emission (kg CO2/kg)       10360 non-null  float64
dtypes: float64(4), int64(2), object(3)
memory usage: 728.6+ KB
None


In [81]:
# Descriptive statistics
print("\nDescriptive Statistics:")
display(df.describe())


Descriptive Statistics:


,Strength (1-10),Weight Capacity (kg),Biodegradability Score (1-10),Recyclability (%),Cost (USD),CO2 Emission (kg CO2/kg)
count,10360.000000,10360.000000,10360.000000,10360.000000,10360.000000,10360.000000
mean,6.544402,12.817612,5.013900,75.337257,2.759958,2.341809
std,2.252254,14.965156,3.508626,10.290592,1.342550,1.494469
min,3.000000,0.100000,1.000000,50.010000,1.000000,0.300000
25%,5.000000,3.010000,2.000000,69.200000,1.920000,1.110000
50%,7.000000,7.205000,3.000000,75.715000,2.500000,1.920000
75%,8.000000,19.110000,9.000000,83.400000,3.130000,3.330000
max,10.000000,99.870000,10.000000,95.000000,9.980000,7.980000


In [82]:
# Missing values check
print("\nMissing Values per Column:")
print(df.isnull().sum())


Missing Values per Column:
Material_Type                    0
Product_Type                     0
Industry                         0
Strength (1-10)                  0
Weight Capacity (kg)             0
Biodegradability Score (1-10)    0
Recyclability (%)                0
Cost (USD)                       0
CO2 Emission (kg CO2/kg)         0
dtype: int64


In [83]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (10360, 9)


In [84]:
#Encoding Categorical Values
df = pd.get_dummies(
    df,
    columns=["Material_Type", "Product_Type", "Industry"],
    drop_first=True
)

## Random Forest (Cost Prediction)

In [85]:
# Features and target selection for Random Forest
X_cost = df.drop("Cost (USD)", axis=1)
y_cost = df["Cost (USD)"]

In [86]:
#train , test set split
X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_cost,
    y_cost,
    test_size=0.2,
    random_state=42
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c,
    y_temp_c,
    test_size=0.5,
    random_state=42
)
print("Train size:", X_train_c.shape)
print("Validation size:", X_val_c.shape)
print("Test size:", X_test_c.shape)

Train size: (8288, 67)
Validation size: (1036, 67)
Test size: (1036, 67)


In [87]:
# saving column names 
cost_feature_columns = X_train_c.columns
joblib.dump(cost_feature_columns, "trained_models/cost_feature_columns.pkl")

['trained_models/cost_feature_columns.pkl']

In [88]:
# feature scaling Random forest
scaler = StandardScaler()

numerical_cols = [
    "Strength (1-10)",
    "Weight Capacity (kg)",
    "Biodegradability Score (1-10)",
    "Recyclability (%)"
]

X_train_c[numerical_cols] = scaler.fit_transform(X_train_c[numerical_cols])
X_val_c[numerical_cols] = scaler.transform(X_val_c[numerical_cols])
X_test_c[numerical_cols] = scaler.transform(X_test_c[numerical_cols])

In [89]:
# Training Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

rf_model.fit(X_train_c, y_train_c)

,n_estimators,200
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [90]:
# Validation Evaluation Random Forest
y_val_pred = rf_model.predict(X_val_c)

print("Validation Results - Random Forest")

print("MAE:", mean_absolute_error(y_val_c, y_val_pred))
print("RMSE:", root_mean_squared_error(y_val_c, y_val_pred))
print("R2:", r2_score(y_val_c, y_val_pred))

Validation Results - Random Forest
MAE: 0.5691507818692362
RMSE: 0.7001980443644783
R2: 0.7499829696341005


In [91]:
# Test evaluation Random Forest
y_test_pred = rf_model.predict(X_test_c)

print("Test Results - Random Forest")

print("MAE:", mean_absolute_error(y_test_c, y_test_pred))
print("RMSE:", root_mean_squared_error(y_test_c, y_test_pred))
print("R2:", r2_score(y_test_c, y_test_pred))

Test Results - Random Forest
MAE: 0.563862138862206
RMSE: 0.690245898335139
R2: 0.7026012964276435


## XGBoost (Co2 Prediction)

In [92]:
# feature and target selection for XGBoost
X_co2 = df.drop("CO2 Emission (kg CO2/kg)", axis=1)
y_co2 = df["CO2 Emission (kg CO2/kg)"]

In [93]:
# train test validation split XGBoost
X_train_co2, X_temp_co2, y_train_co2, y_temp_co2 = train_test_split(
    X_co2,
    y_co2,
    test_size=0.2,
    random_state=42
)

X_val_co2, X_test_co2, y_val_co2, y_test_co2 = train_test_split(
    X_temp_co2,
    y_temp_co2,
    test_size=0.5,
    random_state=42
)

In [94]:
# saving column name
co2_feature_columns = X_train_co2.columns
joblib.dump(co2_feature_columns, "trained_models/co2_feature_columns.pkl")

['trained_models/co2_feature_columns.pkl']

In [95]:
# feature scaling XGBoost
scaler = StandardScaler()

X_train_co2[numerical_cols] = scaler.fit_transform(X_train_co2[numerical_cols])
X_val_co2[numerical_cols] = scaler.transform(X_val_co2[numerical_cols])
X_test_co2[numerical_cols] = scaler.transform(X_test_co2[numerical_cols])

In [96]:
# training XGBoost
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train_co2, y_train_co2)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [97]:
# validation evaluation
y_val_pred_co2 = xgb_model.predict(X_val_co2)

print("Validation Results - XGBoost")
print("MAE:", mean_absolute_error(y_val_co2, y_val_pred_co2))
print("RMSE:", root_mean_squared_error(y_val_co2, y_val_pred_co2))
print("R2:", r2_score(y_val_co2, y_val_pred_co2))

Validation Results - XGBoost
MAE: 0.4567345857735306
RMSE: 0.5790597923477009
R2: 0.8401131571966112


In [98]:
# test evaluation
y_test_pred_co2 = xgb_model.predict(X_test_co2)

print("Test Results - XGBoost")
print("MAE:", mean_absolute_error(y_test_co2, y_test_pred_co2))
print("RMSE:", root_mean_squared_error(y_test_co2, y_test_pred_co2))
print("R2:", r2_score(y_test_co2, y_test_pred_co2))

Test Results - XGBoost
MAE: 0.48933216691937687
RMSE: 0.6191670345982802
R2: 0.8175782456520577


In [99]:
#processed file
df.to_csv("dataset/EcoPackAI_Preprocessed.csv", index=False)

In [100]:
# saving the trained model
pipeline_cost = Pipeline([
    ("scaler", StandardScaler()),
    ("model", rf_model)
])

pipeline_co2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", xgb_model)
])

joblib.dump(rf_model, "trained_models/cost_model.pkl")
joblib.dump(xgb_model, "trained_models/co2_model.pkl")

['trained_models/co2_model.pkl']